# Step 1: Data Ingestion and Preprocessing

Purpose: build one complete, unsplit dataset for Step 2 EDA and later modeling.

Primary output: `data/processed/dataset_001.csv`.

## 0. Notebook Setup and Paths

All paths are resolved relative to the project root so the notebook can be run from the repository or from the `notebooks/` directory.

In [2]:
from pathlib import Path

import pandas as pd


def find_project_root(start: Path | None = None) -> Path:
    """Find the repository root by walking upward to `pyproject.toml`."""
    current = (start or Path.cwd()).resolve()
    for candidate in (current, *current.parents):
        if (candidate / "pyproject.toml").exists():
            return candidate
    raise FileNotFoundError("Could not find project root containing pyproject.toml")


PROJECT_ROOT = find_project_root()
RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

LABELS_PATH = RAW_DIR / "labels.json"
PAIRS_PATH = RAW_DIR / "sampled_pairs_500k.json"
OUTPUT_PATH = PROCESSED_DIR / "dataset_001.csv"

PROJECT_ROOT, LABELS_PATH, PAIRS_PATH, OUTPUT_PATH

(WindowsPath('C:/Users/BS01493/Projects/SBU Europe/Client/GigaAI/Exp-Laser Resonator Beam Alignment Recommandation System'),
 WindowsPath('C:/Users/BS01493/Projects/SBU Europe/Client/GigaAI/Exp-Laser Resonator Beam Alignment Recommandation System/data/raw/labels.json'),
 WindowsPath('C:/Users/BS01493/Projects/SBU Europe/Client/GigaAI/Exp-Laser Resonator Beam Alignment Recommandation System/data/raw/sampled_pairs_500k.json'),
 WindowsPath('C:/Users/BS01493/Projects/SBU Europe/Client/GigaAI/Exp-Laser Resonator Beam Alignment Recommandation System/data/processed/dataset_001.csv'))

## 1. Load Raw Files

Unit 02 will load `labels.json` and `sampled_pairs_500k.json`, inspect shapes, and confirm the expected paths exist.

In [3]:
# Unit 02 implementation.
labels_df = pd.read_json(LABELS_PATH)
pairs_df = pd.read_json(PAIRS_PATH)

print(f"Labels shape: {labels_df.shape}")
print(f"Pairs shape: {pairs_df.shape}")

Labels shape: (3984, 20)
Pairs shape: (500000, 3)


In [4]:
labels_df.head(2)

,Date,Timestamp,Experiment Number,X Axis Gaussian Equation,Y Axis Gaussian Equation,Gaussian Fit % along X,Gaussian Fit % along Y,X Axis Centroid,Y Axis Centroid,Major Axis Beam Width,Minor Axis Beam Width,Effective Diameter,Ellipticity,Iris Position,Z Position,Pitch Position,Yaw Position,Power Measurement,Exposure Time,filename
0,2025-02-27,2026-06-04 10:10:00,Experiment 51,Aexp[-2([x-5791]/1199)^2],Aexp[-2([x-5676]/1369)^2],62.965881,66.755959,159.936142,13.547040,3259.229980,2792.031982,3132.060059,83.315697,3500,0,2.75,2.02,0.000073,200,beamage_Friday_February_28_2025_00_01_04
1,2025-02-27,2026-06-04 10:12:00,Experiment 51,Aexp[-2([x-5780]/1149)^2],Aexp[-2([x-5654]/1127)^2],83.908615,87.743309,156.070419,14.554696,3718.516602,2844.477051,3322.295898,73.876717,3500,0,2.75,2.03,0.000174,200,beamage_Friday_February_28_2025_00_03_05


In [5]:
pairs_df.head(2)

,index1,index2,diff_count
0,0,2662,0
1,0,2783,0


In [6]:
print(labels_df.dtypes)

Date                        datetime64[us]
Timestamp                   datetime64[us]
Experiment Number                      str
X Axis Gaussian Equation               str
Y Axis Gaussian Equation               str
Gaussian Fit % along X             float64
Gaussian Fit % along Y             float64
X Axis Centroid                    float64
Y Axis Centroid                    float64
Major Axis Beam Width              float64
Minor Axis Beam Width              float64
Effective Diameter                 float64
Ellipticity                        float64
Iris Position                        int64
Z Position                           int64
Pitch Position                     float64
Yaw Position                       float64
Power Measurement                  float64
Exposure Time                        int64
filename                               str
dtype: object


## 2. Validate Required Fields

Unit 02 will validate the raw columns needed for joining, candidate features, Gaussian parsing, target synthesis, and metadata retention.

In [7]:
# Validate Required Fields

required_label_fields = [
    "Experiment Number",
    "Iris Position", "Z Position", "Pitch Position", "Yaw Position",
    "X Axis Gaussian Equation", "Y Axis Gaussian Equation",
    "Date", "Timestamp", "filename" # To drop eventually
]

missing_label_fields = [f for f in required_label_fields if f not in labels_df.columns]
assert not missing_label_fields, f"Missing label fields: {missing_label_fields}"

required_pair_fields = ["index1", "index2", "diff_count"]
missing_pair_fields = [f for f in required_pair_fields if f not in pairs_df.columns]
assert not missing_pair_fields, f"Missing pair fields: {missing_pair_fields}"

# Document assumptions:
# 1. pairs_df 'index1' and 'index2' map to labels_df natural integer zero-based index.
assert pairs_df['index1'].max() < len(labels_df), "index1 out of bounds"
assert pairs_df['index2'].max() < len(labels_df), "index2 out of bounds"

print("All required fields are present and valid.")

All required fields are present and valid.


**Schema Assumptions:**
- `labels.json` represents states and incorporates time sequencing inherently referenced by the pairs.
- `index1` and `index2` in `sampled_pairs_500k.json` directly map to the zero-based row index of `labels_df`.
- The controllable positional features (`Iris Position`, `Z Position`, `Pitch Position`, `Yaw Position`) exist as numeric types.
- The `diff_count` tells us how many properties changed between the states.

No unexpected raw-field issues were detected; all expected keys are correctly populated.

## 3. Join Before and After States

Unit 03 will join pairs to beam-state records using `index1` and `index2`.

In [8]:
# Drop date/time/file metadata from the final CSV: Date, Timestamp, filename
cols_to_drop = ["Date", "Timestamp", "filename"]
clean_labels_df = labels_df.drop(columns=cols_to_drop, errors="ignore")

def clean_column_name(name: str) -> str:
    """Standardize column names to snake_case."""
    c = name.strip().lower()
    c = c.replace("%", "percent")
    c = c.replace(" ", "_")
    while "__" in c:
        c = c.replace("__", "_")
    return c

# Map original columns to clean snake_case names
clean_labels_df.columns = [clean_column_name(col) for col in clean_labels_df.columns]

# Join before (index1) and after (index2)
before_df = clean_labels_df.iloc[pairs_df["index1"]].copy().reset_index(drop=True)
after_df = clean_labels_df.iloc[pairs_df["index2"]].copy().reset_index(drop=True)


## 4. Apply Role-Based Column Prefixes

Unit 03 will produce `before_`, `after_`, `target_`, and `meta_` columns while retaining audit metadata.

In [9]:
# Prefix before_ and after_
before_df = before_df.add_prefix("before_")
after_df = after_df.add_prefix("after_")

# Rename experiment number to meta_before_experiment_number / meta_after_experiment_number
before_df = before_df.rename(columns={"before_experiment_number": "meta_before_experiment_number"})
after_df = after_df.rename(columns={"after_experiment_number": "meta_after_experiment_number"})

# Build metadata columns from pairs_df
meta_df = pd.DataFrame({
    "meta_index1": pairs_df["index1"],
    "meta_index2": pairs_df["index2"],
    "meta_diff_count": pairs_df["diff_count"]
})

# Concatenate all parts
joined_df = pd.concat([meta_df, before_df, after_df], axis=1)

print(f"Joined DataFrame shape: {joined_df.shape}")
print("First 2 rows of joined DataFrame:")
display(joined_df.head(2))


Joined DataFrame shape: (500000, 37)
First 2 rows of joined DataFrame:


,meta_index1,meta_index2,meta_diff_count,meta_before_experiment_number,before_x_axis_gaussian_equation,before_y_axis_gaussian_equation,before_gaussian_fit_percent_along_x,before_gaussian_fit_percent_along_y,before_x_axis_centroid,before_y_axis_centroid,...,after_major_axis_beam_width,after_minor_axis_beam_width,after_effective_diameter,after_ellipticity,after_iris_position,after_z_position,after_pitch_position,after_yaw_position,after_power_measurement,after_exposure_time
0,0,2662,0,Experiment 51,Aexp[-2([x-5791]/1199)^2],Aexp[-2([x-5676]/1369)^2],62.965881,66.755959,159.936142,13.54704,...,3082.312988,2198.292969,2757.234619,69.862854,3500,0,2.75,2.02,0.000142,200
1,0,2783,0,Experiment 51,Aexp[-2([x-5791]/1199)^2],Aexp[-2([x-5676]/1369)^2],62.965881,66.755959,159.936142,13.54704,...,3551.434082,2654.001465,3107.364014,72.084946,3500,0,2.75,2.02,0.000076,200


In [10]:
print(joined_df.dtypes)

meta_index1                              int64
meta_index2                              int64
meta_diff_count                          int64
meta_before_experiment_number              str
before_x_axis_gaussian_equation            str
before_y_axis_gaussian_equation            str
before_gaussian_fit_percent_along_x    float64
before_gaussian_fit_percent_along_y    float64
before_x_axis_centroid                 float64
before_y_axis_centroid                 float64
before_major_axis_beam_width           float64
before_minor_axis_beam_width           float64
before_effective_diameter              float64
before_ellipticity                     float64
before_iris_position                     int64
before_z_position                        int64
before_pitch_position                  float64
before_yaw_position                    float64
before_power_measurement               float64
before_exposure_time                     int64
meta_after_experiment_number               str
after_x_axis_

## 5. Parse Gaussian Equation Features

Unit 04 will parse before/after X/Y Gaussian equation strings into numeric center and scale fields, then report parse failures.

In [9]:
# Unit 04 implementation placeholder.

## 6. Synthesize Targets and Changed Labels

Unit 05 will calculate 2-decimal deltas and binary changed labels for Iris, Z, Pitch, and Yaw positions.

In [10]:
# Unit 05 implementation placeholder.

## 7. Leakage Checks and Metadata Cleanup

Unit 05 will drop after-state controllable parameter columns and validate changed-label counts against `meta_diff_count`.

In [11]:
# Unit 05 implementation placeholder.

## 8. Missing-Value Inspection and Resolution

Unit 06 will inspect missing values, document the chosen resolution strategy, and ensure the final dataset has no missing values.

In [12]:
# Unit 06 implementation placeholder.

## 9. Final Dataset Validation

Unit 06 will verify that the dataset is unsplit, contains no raw Gaussian strings, contains no after-state controllable parameter leakage, and is ready for Step 2 EDA.

In [13]:
# Unit 06 implementation placeholder.

## 10. Save Processed CSV

Unit 06 will save exactly one processed CSV to `data/processed/dataset_001.csv`.

In [14]:
# Unit 06 implementation placeholder.
# PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
# final_df.to_csv(OUTPUT_PATH, index=False)

## 11. End-of-Step Review

Unit 07 will summarize validation results, record the Step 1 outcome, and decide whether any tested helper logic should be promoted into `src/`.